# 🛰️ Phase 2: Lahore Sentinel-2 Data Acquisition

This notebook implements a professional data acquisition pipeline using **Google Earth Engine (GEE)** and the `geemap` package. 

**Goal:** Extract Sentinel-2 satellite imagery for the Lahore district and slice it into $64 \times 64$ patches to evaluate the generalization of the EuroSAT-trained models on local Pakistani terrain.

In [ ]:
# --- Libraries and Dependencies ---
# Import necessary libraries for satellite data processing and geospatial visualization
import ee
import geemap
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
# --- Earth Engine Authentication ---
# Initialize Google Earth Engine to access Sentinel-2 data and GEE Assets
# Use Project ID from environment variables for better security and portability
project_id = os.getenv('GEE_PROJECT_ID')

ee.Authenticate(force=True)
ee.Initialize(project=project_id)
print(f"Earth Engine initialized successfully with project: {project_id}")

In [ ]:
ASSET_PATH = "projects/ee-shahidusmaninfo/assets/Lahore_Project/District_Lahore"
roi = ee.FeatureCollection(ASSET_PATH)

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(roi, {"color": "red"}, "Lahore boundary")
Map

In [ ]:
geometry = roi.geometry()

def mask_s2_clouds(image):
    qa = image.select("QA60")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(mask)


In [ ]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(geometry)
    .filterDate("2026-01-01", "2026-03-30")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .map(mask_s2_clouds)
)

print("Number of images:", s2.size().getInfo())

In [ ]:
rgb = s2.median().clip(geometry).select(["B4", "B3", "B2"])

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(rgb, {"min": 0, "max": 3000}, "Lahore RGB")
Map.addLayer(roi, {"color": "yellow"}, "Boundary")
Map

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=rgb,
    description="lahore_rgb_2023",
    folder="lahores2",
    fileNamePrefix="lahore_rgb_2023",
    scale=10,
    region=geometry,
    crs="EPSG:32643",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)
task.start()
print("Export started. Monitor progress at: https://code.earthengine.google.com/tasks")

In [ ]:
import math

# Grid settings
GRID_ROWS = 6
GRID_COLS = 6
PATCH_EXPORT_SIZE = 64  # pixels per patch

# Get bounding box
bounds = geometry.bounds().getInfo()["coordinates"][0]
lon_min = bounds[0][0]
lat_min = bounds[0][1]
lon_max = bounds[2][0]
lat_max = bounds[2][1]

lon_step = (lon_max - lon_min) / GRID_COLS
lat_step = (lat_max - lat_min) / GRID_ROWS

print(f"Total tiles to export: {GRID_ROWS * GRID_COLS}")
print("Starting export tasks to Google Drive folder: lahore_patches_rgb\n")

task_list = []
tile_count = 0

for row in range(GRID_ROWS):
    for col in range(GRID_COLS):
        tile_lon_min = lon_min + col * lon_step
        tile_lon_max = tile_lon_min + lon_step
        tile_lat_min = lat_min + row * lat_step
        tile_lat_max = tile_lat_min + lat_step

        tile_region = ee.Geometry.Rectangle(
            [tile_lon_min, tile_lat_min, tile_lon_max, tile_lat_max]
        )

        tile_img = rgb.clip(tile_region)

        task = ee.batch.Export.image.toDrive(
            image=tile_img,
            description=f"lahore_rgb_r{row:02d}_c{col:02d}",
            folder="lahore_patches_rgb",
            fileNamePrefix=f"lahore_rgb_r{row:02d}_c{col:02d}",
            scale=10,
            region=tile_region,
            crs="EPSG:32643",
            maxPixels=1e9,
            fileFormat="GeoTIFF"
        )
        task.start()
        task_list.append(task)
        tile_count += 1
        print(f"  ✓ Task started: tile row={row} col={col}")

print(f"\n{tile_count} export tasks running in background.")
print("Monitor at: https://code.earthengine.google.com/tasks")
print("Files will appear in Google Drive → lahore_patches_rgb/")